In [ ]:
# M3 PTX kernel (IL-only, no C++, no torch) - 1/3: GPU + mono (KHONG build C/C++)
import os, subprocess, glob, re
!nvidia-smi
r=subprocess.run('apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mono-complete', shell=True, capture_output=True, text=True, timeout=1200)
print('mono install rc=', r.returncode, (r.stderr or '')[-400:])
v=subprocess.run('mono --version', shell=True, capture_output=True, text=True)
print((v.stdout or '').split('\n')[0])
print('KHONG import torch - duong IL-only doc lap')

In [ ]:
# 2/3: clone backend + symlink cudart/cublas/cuda(driver)
import os, subprocess, glob
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('CUDA backend HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD',shell=True).decode().strip())
libdir='/content/tkvlib'; os.makedirs(libdir, exist_ok=True)
def pick(pats):
    for pat in pats:
        found=[p for p in glob.glob(pat) if os.path.exists(os.path.realpath(p))]
        if found: return sorted(found)[0]
    return None
cudart=pick(['/usr/local/cuda*/lib64/libcudart.so','/usr/lib/x86_64-linux-gnu/libcudart.so*'])
cublas=pick(['/usr/local/cuda*/lib64/libcublas.so','/usr/lib/x86_64-linux-gnu/libcublas.so*'])
cudrv=pick(['/usr/lib64-nvidia/libcuda.so*','/usr/lib/x86_64-linux-gnu/libcuda.so*','/usr/local/cuda/compat/libcuda.so*'])
print('cudart=',cudart); print('cublas=',cublas); print('cuda driver=',cudrv)
for nm,tgt in [('cudart.dll',cudart),('cublas.dll',cublas),('cuda.dll',cudrv)]:
    if tgt is None: continue
    p=libdir+'/'+nm
    if os.path.lexists(p): os.remove(p)
    os.symlink(tgt,p); print('symlink',nm,'->',tgt)
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
print('LD_LIBRARY_PATH=',os.environ['LD_LIBRARY_PATH'][:160])

In [ ]:
# 3/3: chay PTX kernel tu .tkv -> IL -> CUDA driver
import subprocess, os, re
exe=d+'/Release/cuda_backend/examples/ilgpu_ptx_scale.exe'
print('exe exists=',os.path.exists(exe),' bytes=',os.path.getsize(exe) if os.path.exists(exe) else 0)
print('co file C++ hay CUDA source sinh?', [p for p in ['ilgpu_ptx_scale.torch.cpp','ilgpu_ptx_scale.cuda.cu','ilgpu_ptx_scale.ptx.cu'] if os.path.exists(os.path.dirname(exe)+'/'+p)])
r=subprocess.run(['mono',exe], env=dict(os.environ), capture_output=True, text=True, timeout=300)
print('mono rc=',r.returncode)
print(r.stdout)
print('STDERR:',(r.stderr or '')[:2000])
m=re.search(r'MAXERR\s+([0-9.eE+-]+)',r.stdout)
print('PARSED PTX MAXERR=',m.group(1) if m else 'KHONG THAY')
print('KET LUAN PTX:', 'OK' if (m and float(m.group(1))<1e-4) else 'XEM OUTPUT')